# EEG Preprocessing Pipeline

**Goal:** Preprocess raw EDF files from CHB-MIT and Siena Scalp into clean, normalized `.npy` epoch arrays for seizure detection and classification.

**Data state at start of this notebook:**
- CHB-MIT: already stored as bipolar channels in raw EDF (`FP1-F7`, `F7-T7`, ...), 256 Hz, 23 channels
- Siena Scalp: bipolar conversion already done and saved as EDF

**Pipeline stages:**
1. Configuration & imports
2. Common bipolar channel selection (intersection of both datasets)
3. Bandpass + notch filtering
4. Resampling to common rate
5. Artifact rejection
6. Normalization
7. Epoching + label assignment
8. Save to `.npy`
9. Dataset sanity check

**Memory strategy:** Files are processed one at a time. Each file is loaded, processed, saved, then immediately freed. The 70 GB dataset never fully enters RAM.

---

## Section 1 — Imports & Global Configuration

In [6]:
import os
import gc
import re
import json
import warnings
import numpy as np
import mne
from pathlib import Path
from scipy.signal import welch
from collections import defaultdict

mne.set_log_level('WARNING')
warnings.filterwarnings('ignore')

# ── Directories ────────────────────────────────────────────────────────────────
CHBMIT_RAW_DIR     = '../data/raw/CHB-MIT'          # raw bipolar EDFs
SEINA_BIPOLAR_DIR  = '../data/processed/Seina'      # your already-converted bipolar EDFs

CHBMIT_OUT_DIR     = '../data/preprocessed/CHB-MIT'
SEINA_OUT_DIR      = '../data/preprocessed/Seina'

# ── Signal parameters ──────────────────────────────────────────────────────────
TARGET_SFREQ   = 256    # Hz  (CHB-MIT is already 256; Siena was 512 → downsample)
WINDOW_SEC     = 4      # seconds per epoch
OVERLAP        = 0.5    # 50% overlap → step = 2 s

# ── Filter parameters ──────────────────────────────────────────────────────────
LOWCUT_HZ      = 0.5    # high-pass cutoff
HIGHCUT_HZ     = 70.0   # low-pass cutoff
NOTCH_CHBMIT   = 60.0   # USA powerline
NOTCH_SEINA    = 50.0   # EU powerline

# ── Artifact thresholds ────────────────────────────────────────────────────────
AMP_THRESH_UV  = 500.0  # µV  — spike / saturation
FLAT_THRESH_UV = 0.5    # µV std — dead / flat channel
EMG_RATIO_THR  = 3.0    # EMG band / beta ratio threshold

# ── Label for ictal overlap fraction ──────────────────────────────────────────
ICTAL_OVERLAP  = 0.5    # epoch labelled ictal if >50% inside a seizure interval

# Create output directories
for d in [CHBMIT_OUT_DIR, SEINA_OUT_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print('✓ Configuration complete')
print(f'  Target sfreq : {TARGET_SFREQ} Hz')
print(f'  Epoch window : {WINDOW_SEC}s  overlap={int(OVERLAP*100)}%')
print(f'  Bandpass     : {LOWCUT_HZ}-{HIGHCUT_HZ} Hz')
print(f'  Amp threshold: ±{AMP_THRESH_UV} µV')

✓ Configuration complete
  Target sfreq : 256 Hz
  Epoch window : 4s  overlap=50%
  Bandpass     : 0.5-70.0 Hz
  Amp threshold: ±500.0 µV


---
## Section 2 — Common Bipolar Channel Definition

CHB-MIT has 23 bipolar channels but some (`T7-FT9`, `FT9-FT10`, `FT10-T8`) are not present in Siena. We keep only the **18 channels common to both datasets** so the feature space is identical for training.

| Chain | Channels |
|---|---|
| Left temporal | FP1-F7, F7-T7, T7-P7, P7-O1 |
| Left parasagittal | FP1-F3, F3-C3, C3-P3, P3-O1 |
| Right parasagittal | FP2-F4, F4-C4, C4-P4, P4-O2 |
| Right temporal | FP2-F8, F8-T8, T8-P8, P8-O2 |
| Midline | FZ-CZ, CZ-PZ |

In [7]:
# Canonical 18-channel set — must exist in every processed file
# CHB-MIT uses these exact names; Siena bipolar files should also use these
# after your bipolar conversion (adjust if your naming differs)
COMMON_CHANNELS = [
    'FP1-F7', 'F7-T7',  'T7-P7',  'P7-O1',   # left temporal
    'FP1-F3', 'F3-C3',  'C3-P3',  'P3-O1',   # left parasagittal
    'FP2-F4', 'F4-C4',  'C4-P4',  'P4-O2',   # right parasagittal
    'FP2-F8', 'F8-T8',  'T8-P8',  'P8-O2',   # right temporal
    'FZ-CZ',  'CZ-PZ',                         # midline
]

# CHB-MIT has duplicate T8-P8 renamed to T8-P8-0 and T8-P8-1 by MNE
# We treat T8-P8-0 as T8-P8 (the standard one) and drop T8-P8-1
CHBMIT_RENAME = {
    'T8-P8-0': 'T8-P8',
    'P7-T7'  : None,      # duplicate of T7-P7, drop
    'T7-FT9' : None,      # not in Siena, drop
    'FT9-FT10': None,
    'FT10-T8': None,
    'T8-P8-1': None,
}

def resolve_channels(raw, dataset='chbmit'):
    """
    Returns a dict {canonical_name: actual_name_in_raw} for
    whichever common channels are present in this file.
    Missing channels are silently skipped.
    """
    available = set(raw.ch_names)
    mapping   = {}  # canonical → actual

    if dataset == 'chbmit':
        # Apply rename table first
        renamed = {}
        for actual in raw.ch_names:
            canonical = CHBMIT_RENAME.get(actual, actual)  # None = drop
            if canonical is not None:
                renamed[canonical] = actual
        for ch in COMMON_CHANNELS:
            if ch in renamed:
                mapping[ch] = renamed[ch]
    else:
        # Siena — expect channels already named like COMMON_CHANNELS
        # but handle case/whitespace differences defensively
        norm = {c.strip().upper(): c for c in raw.ch_names}
        for ch in COMMON_CHANNELS:
            if ch.upper() in norm:
                mapping[ch] = norm[ch.upper()]

    return mapping

print(f'✓ Common channel set defined: {len(COMMON_CHANNELS)} channels')
print(COMMON_CHANNELS)

✓ Common channel set defined: 18 channels
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']


---
## Section 3 — Annotation Parsers

Returns seizure intervals as `[(start_sec, end_sec, seizure_type), ...]` for each EDF file.

- **CHB-MIT:** annotations are in `.seizures` summary files inside each patient folder
- **Siena:** annotations are in per-patient JSON files

In [8]:
# ── CHB-MIT annotation parser ──────────────────────────────────────────────────
def parse_chbmit_annotations(patient_dir: str):
    """
    Reads the chbXX-summary.txt file and returns a dict:
        { 'chb01_03.edf': [(start_sec, end_sec, 'focal'), ...], ... }
    CHB-MIT does not specify seizure type — we default to 'unknown'.
    """
    summary_files = list(Path(patient_dir).glob('*-summary.txt'))
    if not summary_files:
        print(f'  WARNING: No summary file found in {patient_dir}')
        return {}

    annotations = defaultdict(list)
    current_file = None

    with open(summary_files[0], 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('File Name:'):
                current_file = line.split(':')[-1].strip()
            elif line.startswith('Seizure') and 'Start Time' in line and current_file:
                start = int(re.search(r'(\d+)', line).group(1))
            elif line.startswith('Seizure') and 'End Time' in line and current_file:
                end = int(re.search(r'(\d+)', line).group(1))
                annotations[current_file].append((float(start), float(end), 'unknown'))

    return dict(annotations)


# ── Siena annotation parser ────────────────────────────────────────────────────
import re
from datetime import datetime

def parse_siena_annotations(patient_dir: str):
    annotations = defaultdict(list)
    patient_dir = Path(patient_dir)

    txt_files = list(patient_dir.glob('Seizures-list-*.txt'))
    if not txt_files:
        return {}

    def to_sec(timestr):
        """'HH.MM.SS' → total seconds as float"""
        dt = datetime.strptime(timestr.strip(), '%H.%M.%S')
        return dt.hour * 3600 + dt.minute * 60 + dt.second

    for txt_file in txt_files:
        text = txt_file.read_text(errors='replace')

        # Split into seizure blocks on "Seizure n <N>"
        blocks = re.split(r'Seizure\s+n\s+\d+', text, flags=re.IGNORECASE)

        for block in blocks[1:]:   # blocks[0] is the header
            fn_match  = re.search(r'File name:\s*(\S+\.edf)',                    block, re.IGNORECASE)
            reg_match = re.search(r'Registration start time:\s*(\d{2}\.\d{2}\.\d{2})', block, re.IGNORECASE)
            sz_start  = re.search(r'Seizure start time:\s*(\d{2}\.\d{2}\.\d{2})',      block, re.IGNORECASE)
            sz_end    = re.search(r'Seizure end time:\s*(\d{2}\.\d{2}\.\d{2})',        block, re.IGNORECASE)

            if not all([fn_match, reg_match, sz_start, sz_end]):
                print(f'  WARNING: incomplete seizure block in {txt_file.name}, skipping')
                continue

            edf_name       = fn_match.group(1).strip()
            reg_start_sec  = to_sec(reg_match.group(1))
            seiz_start_sec = to_sec(sz_start.group(1))
            seiz_end_sec   = to_sec(sz_end.group(1))

            start = seiz_start_sec - reg_start_sec
            end   = seiz_end_sec   - reg_start_sec

            # Handle midnight rollover
            if start < 0: start += 86400
            if end   < 0: end   += 86400

            annotations[edf_name].append((start, end, 'FBTC'))

    return dict(annotations)


# ── Sanity check ──────────────────────────────────────────────────────────────
raw_pn00 = Path(SEINA_BIPOLAR_DIR).parent.parent / 'raw' / 'SienaScalp' / 'PN00'
ann = parse_siena_annotations(str(raw_pn00))
for fname, seizures in ann.items():
    for start, end, stype in seizures:
        print(f'  {fname}: {start:.1f}s → {end:.1f}s  ({stype})  duration={end-start:.1f}s')
        
# ── Seizure type → integer label map ──────────────────────────────────────────
# 0 = Normal (interictal)
# 1+ = Seizure types from Siena
# CHB-MIT seizures are labelled 'unknown' and map to their own class
SEIZURE_TYPE_MAP = {
    'normal'  : 0,
    'FBTC'    : 1,
    'FIAS'    : 2,
    'FAS'     : 3,
    'GTCS'    : 4,
    'GAS'     : 5,
    'unknown' : 6,
    'UNK'     : 6,
}

def get_epoch_label(start_sec, end_sec, seizure_intervals):
    """
    Returns (is_ictal: bool, seizure_type_label: int)
    is_ictal = True if >50% of epoch overlaps a seizure interval.
    """
    best_overlap = 0.0
    best_type    = 'normal'

    for sz_start, sz_end, sz_type in seizure_intervals:
        overlap = max(0.0, min(end_sec, sz_end) - max(start_sec, sz_start))
        if overlap > best_overlap:
            best_overlap = overlap
            best_type    = sz_type

    is_ictal = best_overlap / (end_sec - start_sec) > ICTAL_OVERLAP
    type_label = SEIZURE_TYPE_MAP.get(best_type, 6) if is_ictal else 0
    return is_ictal, type_label

print('✓ Annotation parsers defined')
print(f'  Seizure type map: {SEIZURE_TYPE_MAP}')

  PN00-1.edf: 1143.0s → 1213.0s  (FBTC)  duration=70.0s
  PN00-2.edf: 1220.0s → 1274.0s  (FBTC)  duration=54.0s
  PN00-3.edf: 765.0s → 4425.0s  (FBTC)  duration=3660.0s
  PN00-4.edf: 1006.0s → 1080.0s  (FBTC)  duration=74.0s
  PN00-5.edf: 904.0s → 971.0s  (FBTC)  duration=67.0s
✓ Annotation parsers defined
  Seizure type map: {'normal': 0, 'FBTC': 1, 'FIAS': 2, 'FAS': 3, 'GTCS': 4, 'GAS': 5, 'unknown': 6, 'UNK': 6}


---
## Section 4 — Signal Processing Functions

All functions operate on a single EDF file's data at a time to keep memory usage low.

In [9]:
# ── 4a. Filtering ──────────────────────────────────────────────────────────────
def apply_filters(raw, notch_hz):
    sfreq   = raw.info['sfreq']
    nyquist = sfreq / 2.0

    l_freq = LOWCUT_HZ
    h_freq = min(HIGHCUT_HZ, nyquist - 2.0)   # always stay below Nyquist

    if h_freq <= l_freq:
        print(f'    WARNING: sfreq={sfreq} Hz too low for bandpass — skipping filter')
        return raw

    raw.filter(
        l_freq=l_freq, h_freq=h_freq,
        method='fir', fir_window='hamming',
        verbose=False
    )

    if notch_hz < nyquist:
        raw.notch_filter(freqs=[notch_hz], verbose=False)
    else:
        print(f'    WARNING: notch {notch_hz} Hz >= Nyquist {nyquist:.1f} Hz — skipping notch')

    return raw


# ── 4b. Resampling ─────────────────────────────────────────────────────────────
def resample_if_needed(raw, allow_upsample=True):
    sfreq = int(raw.info['sfreq'])
    if sfreq == TARGET_SFREQ:
        return raw
    if sfreq < TARGET_SFREQ and not allow_upsample:
        print(f'    WARNING: sfreq={sfreq} Hz < target {TARGET_SFREQ} Hz — skipping resample')
        return raw
    # Upsample or downsample to TARGET_SFREQ
    raw.resample(TARGET_SFREQ, npad='auto', verbose=False)
    return raw

# ── 4c. Artifact rejection ─────────────────────────────────────────────────────
def is_clean_epoch(epoch_uv):
    """
    epoch_uv: np.ndarray (n_channels, n_times) in µV
    Returns True if the epoch passes all artifact checks.
    """
    # Check 1: amplitude spike / saturation
    if np.any(np.max(np.abs(epoch_uv), axis=1) > AMP_THRESH_UV):
        return False

    # Check 2: flat / dead channel
    if np.any(np.std(epoch_uv, axis=1) < FLAT_THRESH_UV):
        return False

    # Check 3: NaN / Inf
    if not np.isfinite(epoch_uv).all():
        return False

    return True


def has_emg_artifact(epoch_uv, sfreq):
    """
    Detects muscle artifact by checking if high-gamma power is
    disproportionately large relative to beta power.
    """
    freqs, psd = welch(epoch_uv, sfreq, nperseg=sfreq, axis=1)

    beta_idx = (freqs >= 13) & (freqs <= 30)
    emg_idx  = (freqs >= 40) & (freqs <= 70)

    beta_pow = psd[:, beta_idx].mean(axis=1)
    emg_pow  = psd[:, emg_idx ].mean(axis=1)

    ratio = emg_pow / (beta_pow + 1e-10)
    return bool(np.any(ratio > EMG_RATIO_THR))


# ── 4d. Normalization ──────────────────────────────────────────────────────────
def normalize_epochs(epochs):
    """
    Z-score per channel using statistics computed across ALL epochs
    in this single recording. Never normalize across patients.

    epochs: np.ndarray (N, n_channels, n_times)  float32
    Returns: normalized array, same shape
    """
    # axis=(0,2) → mean/std over epochs and time, per channel
    mu  = epochs.mean(axis=(0, 2), keepdims=True)   # (1, C, 1)
    std = epochs.std( axis=(0, 2), keepdims=True) + 1e-8
    return ((epochs - mu) / std).astype(np.float32)


print('✓ Signal processing functions defined')
print(f'  Bandpass  : {LOWCUT_HZ}-{HIGHCUT_HZ} Hz')
print(f'  Resample  : → {TARGET_SFREQ} Hz')
print(f'  Amp check : > ±{AMP_THRESH_UV} µV → rejected')
print(f'  Flat check: std < {FLAT_THRESH_UV} µV → rejected')
print(f'  EMG check : emg/beta ratio > {EMG_RATIO_THR} → rejected')

✓ Signal processing functions defined
  Bandpass  : 0.5-70.0 Hz
  Resample  : → 256 Hz
  Amp check : > ±500.0 µV → rejected
  Flat check: std < 0.5 µV → rejected
  EMG check : emg/beta ratio > 3.0 → rejected


---
## Section 5 — Epoch Iterator

Streams epochs one at a time from the data array — never holds all epochs in memory simultaneously during the rejection loop.

In [10]:
def iter_epochs(data_uv, seizure_intervals, sfreq):
    """
    Generator — yields one clean epoch at a time.

    Parameters
    ----------
    data_uv          : np.ndarray (n_channels, n_times) in µV
    seizure_intervals: list of (start_sec, end_sec, seizure_type)
    sfreq            : sampling frequency

    Yields
    ------
    epoch      : np.ndarray (n_channels, win_samples)  float32
    is_ictal   : bool
    type_label : int  (0 = normal, 1-6 = seizure type per SEIZURE_TYPE_MAP)
    """
    win_samples  = int(WINDOW_SEC * sfreq)
    step_samples = int(win_samples * (1.0 - OVERLAP))
    n_times      = data_uv.shape[1]

    total = rejected_amp = rejected_flat = rejected_emg = 0

    for start in range(0, n_times - win_samples + 1, step_samples):
        end   = start + win_samples
        epoch = data_uv[:, start:end].astype(np.float32)
        total += 1

        # Artifact checks
        if np.any(np.max(np.abs(epoch), axis=1) > AMP_THRESH_UV):
            rejected_amp += 1
            continue
        if np.any(np.std(epoch, axis=1) < FLAT_THRESH_UV):
            rejected_flat += 1
            continue
        if has_emg_artifact(epoch, sfreq):
            rejected_emg += 1
            continue

        start_sec  = start / sfreq
        end_sec    = end   / sfreq
        is_ictal, type_label = get_epoch_label(start_sec, end_sec, seizure_intervals)

        yield epoch, is_ictal, type_label

    # Attach rejection stats as generator attributes (accessed after exhaustion)
    iter_epochs._last_stats = {
        'total'       : total,
        'rejected_amp': rejected_amp,
        'rejected_flat': rejected_flat,
        'rejected_emg': rejected_emg,
        'kept'        : total - rejected_amp - rejected_flat - rejected_emg,
    }

print('✓ Epoch iterator defined')
print(f'  Window  : {WINDOW_SEC}s = {int(WINDOW_SEC * TARGET_SFREQ)} samples')
print(f'  Step    : {WINDOW_SEC * (1-OVERLAP)}s = {int(WINDOW_SEC * (1-OVERLAP) * TARGET_SFREQ)} samples')

✓ Epoch iterator defined
  Window  : 4s = 1024 samples
  Step    : 2.0s = 512 samples


---
## Section 6 — Core File Processor

Processes a single EDF file end-to-end:
1. Load → pick channels → filter → resample → load data into RAM
2. Stream epochs → artifact rejection → labelling
3. Normalize → save as `.npy` → free RAM

In [11]:
def process_single_file(edf_path, seizure_intervals, out_dir, dataset='chbmit'):
    edf_path = Path(edf_path)
    stem     = edf_path.stem
    notch_hz = NOTCH_CHBMIT if dataset == 'chbmit' else NOTCH_SEINA

    print(f'  Processing: {edf_path.name}')

    # ── Step A: Load into RAM ──────────────────────────────────────────────────
    try:
        raw = mne.io.read_raw_edf(str(edf_path), preload=True, verbose=False)
    except MemoryError:
        print(f'    ERROR: MemoryError loading {edf_path.name} — file too large')
        return None
    except Exception as e:
        print(f'    ERROR loading file: {e}')
        return None

    # ── Step B: Resolve + pick channels ───────────────────────────────────────
    ch_map = resolve_channels(raw, dataset)
    if len(ch_map) == 0:
        print(f'    SKIP: no common channels found')
        raw.close(); del raw; gc.collect()
        return None

    if len(ch_map) < len(COMMON_CHANNELS):
        missing = set(COMMON_CHANNELS) - set(ch_map.keys())
        print(f'    WARNING: {len(missing)} channels missing → {sorted(missing)}')

    rename_dict = {v: k for k, v in ch_map.items() if v != k}
    if rename_dict:
        raw.rename_channels(rename_dict)
    raw.pick_channels(list(ch_map.keys()), ordered=True)

    # ── Step C: Process (filter + resample) in-place ──────────────────────────
    apply_filters(raw, notch_hz)
    resample_if_needed(raw)

    # ── Step D: Extract numpy array, free MNE object immediately ──────────────
    sfreq       = raw.info['sfreq']
    data_uv     = raw.get_data() * 1e6   # V → µV
    n_ch_actual = data_uv.shape[0]
    raw.close(); del raw; gc.collect()   # ← MNE object gone, only numpy remains

    # ── Step E: Epoch + artifact rejection ────────────────────────────────────
    epochs_list, labels_list = [], []
    for epoch, is_ictal, type_label in iter_epochs(data_uv, seizure_intervals, sfreq):
        epochs_list.append(epoch)
        labels_list.append([int(is_ictal), type_label])

    stats = iter_epochs._last_stats.copy()
    del data_uv; gc.collect()           # ← raw signal gone, only epochs remain

    if not epochs_list:
        print(f'    SKIP: 0 clean epochs after artifact rejection')
        return None

    # ── Step F: Stack + normalize + save ──────────────────────────────────────
    epochs = np.stack(epochs_list, axis=0).astype(np.float32)  # (N, C, T)
    labels = np.array(labels_list, dtype=np.int8)               # (N, 2)
    del epochs_list, labels_list; gc.collect()

    epochs = normalize_epochs(epochs)

    Path(out_dir).mkdir(parents=True, exist_ok=True)
    np.save(f'{out_dir}/{stem}_epochs.npy', epochs)
    np.save(f'{out_dir}/{stem}_labels.npy', labels)

    n_ictal = int(labels[:, 0].sum())
    print(f'    ✓ {len(labels)} epochs saved  '
          f'({n_ictal} ictal, {len(labels)-n_ictal} normal)  '
          f'channels={n_ch_actual}  '
          f'rejected={stats["total"]-stats["kept"]}/{stats["total"]}')

    stats.update({'file': edf_path.name, 'n_epochs': len(labels),
                  'n_ictal': n_ictal, 'n_channels': n_ch_actual})

    del epochs, labels; gc.collect()    # ← epochs gone, only stats dict remains
    return stats

---
## Section 7 — Process CHB-MIT Dataset

CHB-MIT is already stored as bipolar channels in the raw EDFs. We read the summary file for each patient to get seizure intervals, then process each EDF file one at a time.

In [12]:
chbmit_stats = []

patient_dirs = sorted([d for d in Path(CHBMIT_RAW_DIR).iterdir() if d.is_dir()])
print(f'Found {len(patient_dirs)} CHB-MIT patient folders\n')

for patient_dir in patient_dirs:
    patient_id = patient_dir.name
    out_dir    = Path(CHBMIT_OUT_DIR) / patient_id

    print(f'\n── Patient: {patient_id} ──────────────────────────────')

    # Parse all seizure annotations for this patient
    all_annotations = parse_chbmit_annotations(str(patient_dir))

    edf_files = sorted(patient_dir.glob('*.edf'))
    print(f'  {len(edf_files)} EDF files, {len(all_annotations)} with seizures')

    for edf_file in edf_files:
        seizure_intervals = all_annotations.get(edf_file.name, [])

        result = process_single_file(
            edf_path          = edf_file,
            seizure_intervals = seizure_intervals,
            out_dir           = str(out_dir),
            dataset           = 'chbmit'
        )

        if result:
            result['patient'] = patient_id
            chbmit_stats.append(result)

print('\n' + '='*60)
print(f'CHB-MIT complete: {len(chbmit_stats)} files processed')
total_epochs = sum(s['n_epochs'] for s in chbmit_stats)
total_ictal  = sum(s['n_ictal']  for s in chbmit_stats)
print(f'Total epochs : {total_epochs:,}  ({total_ictal:,} ictal, {total_epochs-total_ictal:,} normal)')

Found 24 CHB-MIT patient folders


── Patient: chb01 ──────────────────────────────
  42 EDF files, 7 with seizures
  Processing: chb01_01.edf


OSError: [Errno 22] Invalid argument: '..\\data\\preprocessed\\CHB-MIT\\chb01/chb01_01_epochs.npy'

---
## Section 8 — Process Siena Scalp Dataset

Siena EDFs have already been converted from monopolar to bipolar and saved. We now apply filtering, resampling (512 → 256 Hz), artifact rejection, normalization, and epoching.

In [ ]:
json_files = list(raw_pn00.glob('*.json'))
print(f"\nJSON files found: {len(json_files)}")
if json_files:
    with open(json_files[0]) as f:
        content = json.load(f)
    print(f"\nFirst JSON file: {json_files[0].name}")
    print(json.dumps(content, indent=2))
else:
    # Maybe annotations are in a different format
    print("\nNon-EDF files found:")
    for f in sorted(raw_pn00.iterdir()):
        if not f.suffix == '.edf':
            print(f"  {f.suffix}: {f.name}")

patient_dirs = sorted([d for d in Path(SEINA_BIPOLAR_DIR).iterdir() if d.is_dir()])
print(f'Found {len(patient_dirs)} Siena patient folders\n')


JSON files found: 0

Non-EDF files found:
  .txt: Seizures-list-PN00.txt
Found 14 Siena patient folders



In [ ]:
siena_stats = []

for patient_dir in patient_dirs:
    patient_id = patient_dir.name
    out_dir    = Path(SEINA_OUT_DIR) / patient_id

    print(f'\n── Patient: {patient_id} ──────────────────────────────')

    # Parse Siena JSON annotations for this patient
    # Adjust path if your annotation files are elsewhere
    raw_patient_dir   = Path(SEINA_BIPOLAR_DIR).parent.parent / 'raw' / 'SienaScalp' / patient_id
    all_annotations   = parse_siena_annotations(str(raw_patient_dir))

    edf_files = sorted(patient_dir.glob('*.edf'))
    print(f'  {len(edf_files)} EDF files, {len(all_annotations)} with annotations')

    for edf_file in edf_files:
        seizure_intervals = all_annotations.get(edf_file.name, [])

        result = process_single_file(
            edf_path          = edf_file,
            seizure_intervals = seizure_intervals,
            out_dir           = str(out_dir),
            dataset           = 'siena'
        )

        if result:
            result['patient'] = patient_id
            siena_stats.append(result)

print('\n' + '='*60)
print(f'Siena complete: {len(siena_stats)} files processed')
total_epochs = sum(s['n_epochs'] for s in siena_stats)
total_ictal  = sum(s['n_ictal']  for s in siena_stats)
print(f'Total epochs : {total_epochs:,}  ({total_ictal:,} ictal, {total_epochs-total_ictal:,} normal)')


── Patient: PN00 ──────────────────────────────
  5 EDF files, 5 with annotations
  Processing: PN00-1.edf
    ✓ 1283 epochs saved  (35 ictal, 1248 normal)  channels=12  rejected=28/1311
  Processing: PN00-2.edf
    ✓ 1138 epochs saved  (26 ictal, 1112 normal)  channels=12  rejected=12/1150
  Processing: PN00-3.edf
    ✓ 1175 epochs saved  (805 ictal, 370 normal)  channels=12  rejected=78/1253
  Processing: PN00-4.edf
    ✓ 1022 epochs saved  (31 ictal, 991 normal)  channels=12  rejected=28/1050
  Processing: PN00-5.edf
    ✓ 1064 epochs saved  (31 ictal, 1033 normal)  channels=12  rejected=6/1070

── Patient: PN01 ──────────────────────────────
  1 EDF files, 0 with annotations
  Processing: PN01-1.edf
    ✓ 24167 epochs saved  (0 ictal, 24167 normal)  channels=12  rejected=110/24277

── Patient: PN03 ──────────────────────────────
  2 EDF files, 1 with annotations
  Processing: PN03-1.edf
    ✓ 17448 epochs saved  (0 ictal, 17448 normal)  channels=10  rejected=5869/23317
  Processin

---
## Section 9 — Dataset Sanity Check

Verifies that saved `.npy` files are correct in shape, dtype, value range, and label distribution before moving to training.

In [ ]:
# Quick diagnostic — run this before the main loop
patient_dir = Path(SEINA_BIPOLAR_DIR).parent.parent / 'raw' / 'SienaScalp' / 'PN00'
annotations = parse_siena_annotations(str(patient_dir))

print("Annotation keys:", list(annotations.keys())[:5])

edf_files = sorted((Path(SEINA_BIPOLAR_DIR) / 'PN00').glob('*.edf'))
print("EDF names      :", [f.name for f in edf_files][:5])

def verify_saved_files(processed_root):
    """
    Loads a sample of saved npy files and prints a summary report.
    Uses mmap_mode='r' so files are not fully loaded into RAM.
    """
    epoch_files = sorted(Path(processed_root).rglob('*_epochs.npy'))
    label_files = sorted(Path(processed_root).rglob('*_labels.npy'))

    print(f'Files found: {len(epoch_files)} epoch files, {len(label_files)} label files')

    total_epochs = 0
    total_ictal  = 0
    type_counts  = defaultdict(int)
    shape_errors = []

    expected_time = int(WINDOW_SEC * TARGET_SFREQ)   # 4 * 256 = 1024

    for ef, lf in zip(epoch_files, label_files):
        try:
            ep = np.load(ef, mmap_mode='r')   # (N, C, T)
            lb = np.load(lf, mmap_mode='r')   # (N, 2)

            # Shape checks
            if ep.ndim != 3 or lb.ndim != 2:
                shape_errors.append(f'{ef.name}: unexpected ndim')
                continue
            if ep.shape[2] != expected_time:
                shape_errors.append(f'{ef.name}: time axis={ep.shape[2]}, expected {expected_time}')
            if ep.shape[0] != lb.shape[0]:
                shape_errors.append(f'{ef.name}: epoch/label count mismatch')

            N = ep.shape[0]
            total_epochs += N
            total_ictal  += int(lb[:, 0].sum())

            for t in lb[:, 1]:
                type_counts[int(t)] += 1

        except Exception as e:
            shape_errors.append(f'{ef.name}: {e}')

    # ── Report ─────────────────────────────────────────────────────────────────
    print(f'\nTotal epochs  : {total_epochs:,}')
    print(f'Ictal epochs  : {total_ictal:,}  ({100*total_ictal/max(total_epochs,1):.1f}%)')
    print(f'Normal epochs : {total_epochs - total_ictal:,}')
    print(f'Imbalance ratio: 1 : {(total_epochs-total_ictal)/max(total_ictal,1):.1f}  (ictal : normal)')

    inv_map = {v: k for k, v in SEIZURE_TYPE_MAP.items()}
    print('\nSeizure type distribution:')
    for code, count in sorted(type_counts.items()):
        label = inv_map.get(code, '?')
        bar   = '█' * min(40, int(40 * count / max(total_epochs, 1)))
        print(f'  {code} {label:8s}: {count:6,}  {bar}')

    if shape_errors:
        print(f'\n⚠ Shape/load errors ({len(shape_errors)}):')
        for e in shape_errors:
            print(f'  {e}')
    else:
        print('\n✓ All files passed shape checks')


print('── CHB-MIT preprocessed ──────────────────────────────────')
verify_saved_files(CHBMIT_OUT_DIR)

print('\n── Siena preprocessed ────────────────────────────────────')
verify_saved_files(SEINA_OUT_DIR)

Annotation keys: ['PN00-1.edf', 'PN00-2.edf', 'PN00-3.edf', 'PN00-4.edf', 'PN00-5.edf']
EDF names      : ['PN00-1.edf', 'PN00-2.edf', 'PN00-3.edf', 'PN00-4.edf', 'PN00-5.edf']
── CHB-MIT preprocessed ──────────────────────────────────
Files found: 683 epoch files, 683 label files

Total epochs  : 1,399,433
Ictal epochs  : 309  (0.0%)
Normal epochs : 1,399,124
Imbalance ratio: 1 : 4527.9  (ictal : normal)

Seizure type distribution:
  0 normal  : 1,399,124  ███████████████████████████████████████
  6 UNK     :    309  

✓ All files passed shape checks

── Siena preprocessed ────────────────────────────────────
Files found: 41 epoch files, 41 label files

Total epochs  : 232,762
Ictal epochs  : 1,583  (0.7%)
Normal epochs : 231,179
Imbalance ratio: 1 : 146.0  (ictal : normal)

Seizure type distribution:
  0 normal  : 231,179  ███████████████████████████████████████
  1 FBTC    :  1,583  

✓ All files passed shape checks


---
## Section 10 — Memory-Safe Dataset Loader (for Training)

Use this in your training notebook. Files are memory-mapped — only the batch being used at any moment is in RAM.

In [ ]:
class EEGDatasetLoader:
    """
    Lazily indexes all .npy epoch files across both datasets.
    Compatible with PyTorch DataLoader and plain numpy batching.

    Usage:
        ds = EEGDatasetLoader([CHBMIT_OUT_DIR, SEINA_OUT_DIR])
        epoch, binary_label, type_label = ds[0]
    """

    def __init__(self, root_dirs, label_col='binary'):
        """
        root_dirs  : list of preprocessed root directories
        label_col  : 'binary' (col 0) or 'type' (col 1)
        """
        self.label_col = 0 if label_col == 'binary' else 1

        epoch_files = []
        for root in root_dirs:
            epoch_files += sorted(Path(root).rglob('*_epochs.npy'))

        # Memory-map all files — no data in RAM yet
        self._epochs = [np.load(f, mmap_mode='r') for f in epoch_files]
        self._labels = [
            np.load(str(f).replace('_epochs', '_labels'), mmap_mode='r')
            for f in epoch_files
        ]

        # Flat index: (file_idx, epoch_idx)
        self._index = [
            (fi, ei)
            for fi, ep in enumerate(self._epochs)
            for ei in range(len(ep))
        ]

        print(f'Dataset ready: {len(self._index):,} epochs across {len(epoch_files)} files')

    def __len__(self):
        return len(self._index)

    def __getitem__(self, idx):
        fi, ei = self._index[idx]
        epoch  = np.array(self._epochs[fi][ei]).copy()
        binary = int(self._labels[fi][ei, 0])
        stype  = int(self._labels[fi][ei, 1])

        # Augment ictal epochs only — preserves real signal structure
        if binary == 1:
            epoch = self._augment(epoch)

        return epoch, binary, stype

    def _augment(self, epoch):
        """Light augmentations that preserve seizure morphology."""
        # 1. Gaussian noise (SNR-preserving, very small)
        noise = np.random.normal(0, 0.01, epoch.shape).astype(np.float32)
        epoch = epoch + noise

        # 2. Amplitude scaling ±10%
        scale = np.random.uniform(0.90, 1.10)
        epoch = epoch * scale

        # 3. Time shift ±10% of window (circular)
        shift = np.random.randint(-100, 100)   # samples at 256Hz
        epoch = np.roll(epoch, shift, axis=1)

        return epoch.astype(np.float32)
    
    def get_class_weights(self):
        """Returns inverse-frequency weights for class balancing in loss functions."""
        all_labels = np.concatenate([lb[:, self.label_col] for lb in self._labels])
        classes, counts = np.unique(all_labels, return_counts=True)
        weights = 1.0 / counts
        weights /= weights.sum()
        return dict(zip(classes.tolist(), weights.tolist()))
    
    def get_sampler(self):
        """
        Returns a WeightedRandomSampler so each batch is ~50/50 ictal/normal.
        Use this as the 'sampler' argument in DataLoader.
        """
        from torch.utils.data import WeightedRandomSampler

        all_labels = np.concatenate([lb[:, 0] for lb in self._labels])
        class_counts = np.bincount(all_labels.astype(int))
        # Weight each sample inversely proportional to its class frequency
        sample_weights = np.where(all_labels == 1,
                                1.0 / class_counts[1],
                                1.0 / class_counts[0])
        return WeightedRandomSampler(
            weights     = torch.tensor(sample_weights, dtype=torch.float32),
            num_samples = len(sample_weights),
            replacement = True
        )


# Quick test — comment out if running sequentially for the first time
# ds = EEGDatasetLoader([CHBMIT_OUT_DIR, SEINA_OUT_DIR])
# epoch, binary_label, type_label = ds[0]
# print(f'Sample epoch shape : {epoch.shape}')          # (n_channels, 1024)
# print(f'Binary label       : {binary_label}')         # 0 or 1
# print(f'Type label         : {type_label}')           # 0-6
# print(f'Class weights      : {ds.get_class_weights()}')

print('✓ EEGDatasetLoader defined — use in training notebook')

✓ EEGDatasetLoader defined — use in training notebook


In [ ]:
# In your training notebook
ds = EEGDatasetLoader([CHBMIT_OUT_DIR, SEINA_OUT_DIR])
weights = ds.get_class_weights()
print(weights)  # e.g., {0: 0.00022, 1: 0.99978}

# PyTorch
import torch
import torch.nn as nn

weight_tensor = torch.tensor([weights[0], weights[1]], dtype=torch.float32)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(weights[1] / weights[0]))
# or for multi-class (seizure type):
criterion = nn.CrossEntropyLoss(weight=weight_tensor)

Dataset ready: 1,632,195 epochs across 724 files
{0: 0.0011591752210979693, 1: 0.9988408247789021}


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets.float(), reduction='none'
        )
        pt    = torch.exp(-bce)                          # probability of correct class
        focal = self.alpha * (1 - pt) ** self.gamma * bce
        return focal.mean()

criterion = FocalLoss(alpha=0.25, gamma=2.0)